# 24 — Practice: Module 5 — Advanced Pandas

20 interview-style questions on performance, MultiIndex, method chaining,
and production pipeline design.

In [ ]:
import pandas as pd
import numpy as np
import time

np.random.seed(42)
n = 50_000

# Transaction dataset for performance work
transactions = pd.DataFrame({
    'txn_id':    range(1, n + 1),
    'customer':  np.random.randint(1, 5000, n),
    'product':   np.random.choice(['Laptop', 'Phone', 'Tablet', 'Headphones', 'Charger'], n),
    'category':  np.random.choice(['Electronics', 'Accessories', 'Wearables'], n),
    'amount':    np.round(np.random.uniform(200, 80000, n), 2),
    'quantity':  np.random.randint(1, 10, n),
    'discount':  np.round(np.random.uniform(0, 0.25, n), 3),
    'returned':  np.random.choice([0, 1], n, p=[0.92, 0.08]),
    'region':    np.random.choice(['North', 'South', 'East', 'West'], n),
    'date':      pd.date_range('2022-01-01', periods=n, freq='h')[:n]
})

print(f"Transactions: {transactions.shape}")

---
### Q1: Show iterrows() is slow — replace with vectorized computation

In [ ]:
small = transactions.head(5000).copy()

start = time.perf_counter()
net_revs = []
for _, row in small.iterrows():
    net_revs.append(row['amount'] * row['quantity'] * (1 - row['discount']))
small['net_revenue_slow'] = net_revs
t_slow = time.perf_counter() - start

start = time.perf_counter()
small['net_revenue_fast'] = small['amount'] * small['quantity'] * (1 - small['discount'])
t_fast = time.perf_counter() - start

print(f"iterrows:   {t_slow:.3f}s")
print(f"vectorized: {t_fast:.4f}s")
print(f"Speedup:    {t_slow / t_fast:.0f}×")

---
### Q2: Replace apply(axis=1) with np.where for a conditional column

In [ ]:
# SLOW
start = time.perf_counter()
_ = small.apply(lambda row: 'High' if row['amount'] > 30000 else
                              ('Mid' if row['amount'] > 10000 else 'Low'), axis=1)
t_apply = time.perf_counter() - start

# FAST
start = time.perf_counter()
_ = np.where(small['amount'] > 30000, 'High',
    np.where(small['amount'] > 10000, 'Mid', 'Low'))
t_npwhere = time.perf_counter() - start

print(f"apply:    {t_apply:.3f}s")
print(f"np.where: {t_npwhere:.4f}s")
print(f"Speedup:  {t_apply / t_npwhere:.0f}×")

---
### Q3: Use map() instead of apply() for dict-based lookup

In [ ]:
region_code = {'North': 'N', 'South': 'S', 'East': 'E', 'West': 'W'}

start = time.perf_counter()
_ = transactions['region'].apply(lambda x: region_code.get(x))
t_apply = time.perf_counter() - start

start = time.perf_counter()
_ = transactions['region'].map(region_code)
t_map = time.perf_counter() - start

print(f"apply:  {t_apply:.3f}s")
print(f"map:    {t_map:.4f}s")
print(f"Speedup: {t_apply / t_map:.0f}×")

---
### Q4: Use query() and eval() on a large DataFrame

In [ ]:
min_amount = 10000

start = time.perf_counter()
r1 = transactions[(transactions['amount'] > min_amount) & (transactions['returned'] == 0)]
t_mask = time.perf_counter() - start

start = time.perf_counter()
r2 = transactions.query('amount > @min_amount and returned == 0')
t_query = time.perf_counter() - start

print(f"mask:    {t_mask:.3f}s  rows: {len(r1):,}")
print(f"query(): {t_query:.3f}s  rows: {len(r2):,}")

---
### Q5: Optimize dtype at read time — show memory comparison

In [ ]:
mem_naive = transactions.memory_usage(deep=True).sum() / 1024**2

transactions_opt = transactions.astype({
    'txn_id':   'int32',
    'customer': 'int32',
    'product':  'category',
    'category': 'category',
    'amount':   'float32',
    'quantity': 'int8',
    'discount': 'float32',
    'returned': 'int8',
    'region':   'category',
})

mem_opt = transactions_opt.memory_usage(deep=True).sum() / 1024**2
print(f"Naive: {mem_naive:.1f} MB  Optimized: {mem_opt:.1f} MB  ({(1-mem_opt/mem_naive):.1%} reduction)")

---
### Q6: Create a MultiIndex from product × region × year

In [ ]:
products = ['Laptop', 'Phone', 'Tablet']
regions  = ['North', 'South']
years    = [2022, 2023]

mi = pd.MultiIndex.from_product([products, regions, years],
                                  names=['product', 'region', 'year'])
df_mi = pd.DataFrame({'revenue': np.random.randint(100000, 500000, len(mi))}, index=mi)
print(df_mi)

---
### Q7: Select all rows for 'Laptop' using loc on MultiIndex

In [ ]:
laptop = df_mi.loc['Laptop']
print(laptop)

---
### Q8: Select 2023 rows using xs() on the 'year' level

In [ ]:
year_2023 = df_mi.xs(2023, level='year')
print(year_2023)

---
### Q9: unstack year to columns, compute YoY growth

In [ ]:
wide = df_mi['revenue'].unstack('year')
wide['yoy_growth'] = ((wide[2023] - wide[2022]) / wide[2022] * 100).round(1)
print(wide)

---
### Q10: Flatten a MultiIndex after groupby

In [ ]:
multi_agg = transactions.groupby(['region', 'category']).agg(
    orders=('txn_id', 'count'), revenue=('amount', 'sum')
)
flat = multi_agg.reset_index()
print(flat.head(8))

---
### Q11: Build a full ETL chain using assign + query + groupby

In [ ]:
result = (
    transactions
    .assign(
        net_revenue = lambda df: df['amount'] * df['quantity'] * (1 - df['discount']),
        tier        = lambda df: pd.cut(df['amount'], bins=[0, 5000, 20000, 80001],
                                        labels=['Low', 'Mid', 'High']),
        year        = lambda df: df['date'].dt.year
    )
    .query('returned == 0')
    .groupby(['region', 'category', 'year'])
    .agg(orders=('txn_id', 'count'), total_net=('net_revenue', 'sum'))
    .round(0)
)
print(result.head(12))

---
### Q12: Use pipe() to insert a debug checkpoint

In [ ]:
def log(df, label):
    print(f"[{label}] {df.shape}")
    return df

result = (
    transactions
    .pipe(log, 'start')
    .query('amount > 5000')
    .pipe(log, 'after filter')
    .assign(net=lambda df: df['amount'] * df['quantity'])
    .pipe(log, 'with net')
)

---
### Q13: pd.cut() to bin amount into Low/Mid/High/Premium

In [ ]:
transactions['tier'] = pd.cut(
    transactions['amount'],
    bins=[0, 2000, 10000, 30000, 80001],
    labels=['Low', 'Mid', 'High', 'Premium']
)
print(transactions['tier'].value_counts())

---
### Q14: pd.qcut() for equal-frequency quartile binning

In [ ]:
transactions['amount_quartile'] = pd.qcut(
    transactions['amount'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4']
)
print(transactions['amount_quartile'].value_counts())

---
### Q15: swaplevel + sort_index on a MultiIndex

In [ ]:
swapped = df_mi.swaplevel('product', 'region').sort_index()
print(swapped.head(6))

---
### Q16: Build a production-quality data cleaning pipeline using pipe

In [ ]:
def optimize_types(df):
    return df.astype({'amount': 'float32', 'quantity': 'int8', 'returned': 'int8'})

def add_features(df):
    return df.assign(
        net_revenue = lambda d: d['amount'] * d['quantity'] * (1 - d['discount']),
        year        = lambda d: d['date'].dt.year,
        month       = lambda d: d['date'].dt.month,
    )

def remove_returns(df):
    return df.query('returned == 0')

pipeline_result = (
    transactions
    .pipe(optimize_types)
    .pipe(add_features)
    .pipe(remove_returns)
    .reset_index(drop=True)
)
print(f"Pipeline result: {pipeline_result.shape}")
print(pipeline_result.dtypes)

---
### Q17: MultiIndex groupby at specific level

In [ ]:
# Sum revenue at 'product' level (aggregating across region and year)
by_product = df_mi.groupby(level='product')['revenue'].sum()
print(by_product)

---
### Q18: Flatten MultiLevel column names after pivot_table

In [ ]:
pt = pd.pivot_table(
    transactions, values=['amount', 'quantity'],
    index='region', columns='category', aggfunc='sum'
)
print("MultiLevel columns:", pt.columns.tolist()[:4])

pt.columns = [f'{v}_{c}' for v, c in pt.columns]
print("Flattened:", pt.columns.tolist()[:4])
print(pt)

---
### Q19: Use pd.IndexSlice to select from MultiIndex

In [ ]:
idx_slice = pd.IndexSlice

# Select Phone and Tablet in North and South
subset = df_mi.sort_index().loc[idx_slice[['Phone', 'Tablet'], 'North', :], :]
print(subset)

---
### Q20: Full interview question — top 3 customers by revenue per region

In [ ]:
top3_cust_by_region = (
    transactions
    .assign(revenue=lambda df: df['amount'] * df['quantity'])
    .query('returned == 0')
    .groupby(['region', 'customer'])
    .agg(total_revenue=('revenue', 'sum'), orders=('txn_id', 'count'))
    .reset_index()
    .sort_values(['region', 'total_revenue'], ascending=[True, False])
    .groupby('region', group_keys=False)
    .apply(lambda g: g.nlargest(3, 'total_revenue'))
    .reset_index(drop=True)
    .assign(region_rank=lambda df: df.groupby('region').cumcount() + 1)
)
print(top3_cust_by_region)